# 10 - Verify Data Splits

This notebook verifies the quality and integrity of the data splits created by the `09_create_data_splits.ipynb` notebook.

**Verification Checks**:
1.  **Basic Statistics**: Calculate the number of samples, unique labels, and unique signers in each split (`train`, `val`, `test`).
2.  **Signer Overlap**: Programmatically confirm that there is zero overlap of `signer_id` between the splits. This is crucial for ensuring the model is tested on unseen individuals.
3.  **Class Distribution**: Visualize the percentage distribution of the top classes across the three splits to confirm that stratification was successful.
4.  **Signer Distribution**: Analyze the distribution of samples per signer within each split to check for any potential imbalances.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
plt.style.use('ggplot')

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

# This is the output file from notebook 09
INPUT_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_with_splits.csv'

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV}\\nPlease run notebook '09_create_data_splits.ipynb' first.")

print('INPUT_CSV:', INPUT_CSV)

## 1. Load Data and Calculate Basic Statistics

In [ ]:
df = pd.read_csv(INPUT_CSV)

print("Shape of the dataset:", df.shape)

# Calculate statistics for each split
split_stats = df.groupby('split').agg(
    n_samples=('label', 'count'),
    n_unique_labels=('label', 'nunique'),
    n_unique_signers=('signer_id', 'nunique')
).reset_index()

# Add percentage of samples
total_samples = len(df)
split_stats['sample_percentage'] = (split_stats['n_samples'] / total_samples) * 100

print("\\n--- Basic Split Statistics ---")
display(split_stats)

## 2. Verify Signer Overlap

This is the most critical check. We must ensure that the sets of signers in the `train`, `val`, and `test` splits are completely disjoint.

In [ ]:
train_signers = set(df[df['split'] == 'train']['signer_id'].unique())
val_signers = set(df[df['split'] == 'val']['signer_id'].unique())
test_signers = set(df[df['split'] == 'test']['signer_id'].unique())

print("--- Signer Overlap Check ---")
print(f"Number of signers in train set: {len(train_signers)}")
print(f"Number of signers in val set:   {len(val_signers)}")
print(f"Number of signers in test set:  {len(test_signers)}")
print("-" * 30)

# Check for intersections
train_val_overlap = len(train_signers.intersection(val_signers))
train_test_overlap = len(train_signers.intersection(test_signers))
val_test_overlap = len(val_signers.intersection(test_signers))

print(f"Overlap between train and val sets:  {train_val_overlap}")
print(f"Overlap between train and test sets: {train_test_overlap}")
print(f"Overlap between val and test sets:   {val_test_overlap}")

if train_val_overlap == 0 and train_test_overlap == 0 and val_test_overlap == 0:
    print("\\n✅ Verification successful: No signer overlap detected between any of the sets.")
else:
    print("\\n❌ Verification FAILED: Signer overlap detected! The splits are not valid for this strategy.")

## 3. Analyze Class Distribution Across Splits

Let's visualize the distribution of the top N classes to visually confirm that stratification worked as expected. The percentages for each class should be roughly similar across `train`, `val`, and `test`.

In [ ]:
# Get the overall top 20 most frequent labels
top_20_labels = df['label'].value_counts().nlargest(20).index

# Filter the dataframe to only include these top labels for cleaner visualization
df_top_labels = df[df['label'].isin(top_20_labels)]

# Calculate the percentage of each label within each split
class_dist = (df_top_labels.groupby('split')['label']
              .value_counts(normalize=True)
              .mul(100)
              .rename('percentage')
              .reset_index())

# Create the plot
plt.figure(figsize=(18, 8))
sns.barplot(x='label', y='percentage', hue='split', data=class_dist)
plt.title('Class Distribution Across Splits (Top 20 Classes)', fontsize=16)
plt.xlabel('Label', fontsize=12)
plt.ylabel('Percentage within split (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Split')
plt.tight_layout()
plt.show()

## 4. Analyze Signer Distribution

This analysis helps us understand if any single signer dominates a particular split. Ideally, the samples should be spread across many signers.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 15), sharex=True)
fig.suptitle('Distribution of Samples per Signer in Each Split', fontsize=18)

splits_to_plot = ['train', 'val', 'test']

for i, split_name in enumerate(splits_to_plot):
    ax = axes[i]
    split_df = df[df['split'] == split_name]
    signer_counts = split_df['signer_id'].value_counts().sort_values(ascending=False)
    
    # Plot only the top 50 signers for readability if there are too many
    if len(signer_counts) > 50:
        signer_counts = signer_counts.head(50)
        ax.set_title(f'{split_name.capitalize()} Set (Top 50 Signers)', fontsize=14)
    else:
        ax.set_title(f'{split_name.capitalize()} Set', fontsize=14)
        
    sns.barplot(x=signer_counts.index, y=signer_counts.values, ax=ax, palette='viridis')
    ax.set_ylabel('Number of Samples', fontsize=12)
    ax.tick_params(axis='x', rotation=90, labelsize=8)

axes[-1].set_xlabel('Signer ID', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### Summary of Signer Distribution

Let's look at the descriptive statistics for the number of samples per signer in each split.

In [ ]:
signer_dist_summary = df.groupby('split')['signer_id'].value_counts().groupby('split').describe()

print("--- Descriptive Statistics for Samples per Signer ---")
display(signer_dist_summary)